<a href="https://colab.research.google.com/github/grettmon/UAS-PCP/blob/main/UAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
#dengan rumus
# ============================================================
#   🎞️  FILM ROLL CONVERTER
#   Sistem Pengolahan Citra — Konversi Negatif Film Analog
#
#   Library  : OpenCV, NumPy, Matplotlib, Gradio
#   Platform : Google Colab
#
#   Konsep yang diterapkan:
#     • Aritmatika    — Brightness, Contrast, Channel RGB
#     • Operasi Titik — Negasi, Gamma, Saturation, LUT
#     • Geometri      — Rotasi, Flip, Crop
#     • Spasial       — Denoise, Sharpen, Vignette, Blur
# ============================================================


# ─────────────────────────────────────────────
# CELL 1 — INSTALL & IMPORT
# ─────────────────────────────────────────────

import subprocess
subprocess.run(["pip", "install", "gradio", "opencv-python-headless", "matplotlib", "numpy", "-q"])

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
import io
from PIL import Image


# ─────────────────────────────────────────────
# CELL 2 — FUNGSI GEOMETRI
# ─────────────────────────────────────────────

def rotate_image(img, angle):
    if angle == 0:
        return img
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(img, M, (w, h),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REFLECT)
    return rotated


def flip_image(img, flip_h, flip_v):
    result = img.copy()
    if flip_h and flip_v:
        result = cv2.flip(result, -1)
    elif flip_h:
        result = cv2.flip(result, 1)
    elif flip_v:
        result = cv2.flip(result, 0)
    return result


def crop_image(img, top_pct, bottom_pct, left_pct, right_pct):
    h, w = img.shape[:2]
    top    = int(h * top_pct / 100)
    bottom = int(h * (1 - bottom_pct / 100))
    left   = int(w * left_pct / 100)
    right  = int(w * (1 - right_pct / 100))
    if bottom > top and right > left:
        return img[top:bottom, left:right]
    return img


def zoom_image(img, zoom_factor):
    if zoom_factor == 100:
        return img
    h, w = img.shape[:2]
    alpha = zoom_factor / 100.0

    if alpha > 1.0:
        new_h, new_w = int(h / alpha), int(w / alpha)
        dy = (h - new_h) // 2
        dx = (w - new_w) // 2
        cropped = img[dy:dy+new_h, dx:dx+new_w]
        return cv2.resize(cropped, (w, h), interpolation=cv2.INTER_CUBIC)
    else:
        new_h, new_w = int(h * alpha), int(w * alpha)
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas = np.zeros_like(img)
        dy = (h - new_h) // 2
        dx = (w - new_w) // 2
        canvas[dy:dy+new_h, dx:dx+new_w] = resized
        return canvas


def translate_image(img, shift_x, shift_y):
    if shift_x == 0 and shift_y == 0:
        return img
    h, w = img.shape[:2]
    dx = int(w * shift_x / 100.0)
    dy = int(h * shift_y / 100.0)
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    shifted = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    return shifted


# ─────────────────────────────────────────────
# CELL 3 — FUNGSI ARITMATIKA & OPERASI TITIK
# ─────────────────────────────────────────────

def remove_orange_mask(img, strength=0.8):
    img_float = img.astype(np.float32)
    result = img_float.copy()
    for ch in range(3):
        channel = img_float[:, :, ch]
        p_low  = np.percentile(channel, 1)
        p_high = np.percentile(channel, 99)
        if p_high > p_low:
            corrected = (channel - p_low) / (p_high - p_low) * 255.0
            result[:, :, ch] = (1 - strength) * channel + strength * corrected
    return np.clip(result, 0, 255).astype(np.uint8)


def apply_negasi(img, aktif=True, intensitas=100):
    if not aktif:
        return img
    negasi = 255 - img.astype(np.float32)
    alpha  = intensitas / 100.0
    result = alpha * negasi + (1 - alpha) * img.astype(np.float32)
    return np.clip(result, 0, 255).astype(np.uint8)


def adjust_brightness(img, value):
    img_float = img.astype(np.float32)
    result    = img_float + value
    return np.clip(result, 0, 255).astype(np.uint8)


def adjust_contrast(img, value):
    alpha     = 1.0 + (value / 100.0)
    img_float = img.astype(np.float32)
    result    = alpha * (img_float - 128) + 128
    return np.clip(result, 0, 255).astype(np.uint8)


def adjust_gamma(img, gamma=1.0):
    if gamma == 1.0:
        return img
    inv_gamma = 1.0 / gamma
    lut = np.array([
        ((i / 255.0) ** inv_gamma) * 255
        for i in range(256)
    ], dtype=np.uint8)
    return cv2.LUT(img, lut)


def adjust_temperature(img, value):
    img_float = img.astype(np.float32)
    result    = img_float.copy()
    shift     = abs(value) * 0.5
    if value > 0:
        result[:, :, 2] += shift
        result[:, :, 0] -= shift
    else:
        result[:, :, 2] -= shift
        result[:, :, 0] += shift
    return np.clip(result, 0, 255).astype(np.uint8)


def adjust_saturation(img, value):
    hsv   = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    alpha = 1.0 + (value / 100.0)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * alpha, 0, 255)
    result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    return result


def adjust_rgb_channels(img, r_val, g_val, b_val):
    img_float = img.astype(np.float32)
    result    = img_float.copy()
    result[:, :, 2] += r_val
    result[:, :, 1] += g_val
    result[:, :, 0] += b_val
    return np.clip(result, 0, 255).astype(np.uint8)


def adjust_highlights_shadows(img, highlights=0, shadows=0):
    img_float = img.astype(np.float32)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    hi_mask = np.clip((gray - 0.5) * 2, 0, 1)[:, :, np.newaxis]
    sh_mask = np.clip((0.5 - gray) * 2, 0, 1)[:, :, np.newaxis]
    result = img_float + (highlights * hi_mask) + (shadows * sh_mask)
    return np.clip(result, 0, 255).astype(np.uint8)


def auto_white_balance(img):
    img_float = img.astype(np.float32)
    mean_b = np.mean(img_float[:, :, 0])
    mean_g = np.mean(img_float[:, :, 1])
    mean_r = np.mean(img_float[:, :, 2])
    mean_gray = (mean_b + mean_g + mean_r) / 3.0
    if mean_b > 0: img_float[:, :, 0] *= (mean_gray / mean_b)
    if mean_g > 0: img_float[:, :, 1] *= (mean_gray / mean_g)
    if mean_r > 0: img_float[:, :, 2] *= (mean_gray / mean_r)
    return np.clip(img_float, 0, 255).astype(np.uint8)


def histogram_equalization(img):
    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
    return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)


# ─────────────────────────────────────────────
# CELL 4 — FUNGSI SPASIAL
# ─────────────────────────────────────────────

def apply_denoise(img, strength=0):
    if strength == 0:
        return img
    h = int(strength)
    h = max(1, min(h, 20))
    return cv2.fastNlMeansDenoisingColored(img, None, h, h, 7, 21)


def apply_sharpen(img, strength=0):
    if strength == 0:
        return img
    alpha     = strength / 5.0
    blurred   = cv2.GaussianBlur(img, (0, 0), 3)
    sharpened = cv2.addWeighted(img, 1 + alpha, blurred, -alpha, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def apply_blur(img, strength=0):
    if strength == 0:
        return img
    k = int(strength) * 2 + 1
    return cv2.GaussianBlur(img, (k, k), 0)


def apply_vignette(img, strength=0):
    if strength == 0:
        return img
    h, w   = img.shape[:2]
    sigma  = (1 - strength / 150.0) * max(h, w)
    sigma  = max(sigma, 50)
    kernel_x = cv2.getGaussianKernel(w, sigma)
    kernel_y = cv2.getGaussianKernel(h, sigma)
    mask     = kernel_y * kernel_x.T
    mask     = mask / mask.max()
    mask_3ch = np.stack([mask] * 3, axis=-1)
    result = img.astype(np.float32) * mask_3ch
    return np.clip(result, 0, 255).astype(np.uint8)


# ─────────────────────────────────────────────
# CELL 5 — PIPELINE UTAMA
# ─────────────────────────────────────────────

def pipeline_otomatis(img_bgr):
    result = img_bgr.copy()
    result = remove_orange_mask(result, strength=0.85)
    result = apply_negasi(result, aktif=True, intensitas=100)
    result = auto_white_balance(result)
    result = adjust_contrast(result, value=15)
    result = adjust_brightness(result, value=5)
    result = apply_denoise(result, strength=5)
    result = apply_sharpen(result, strength=3)
    return result


def pipeline_manual(
    img_bgr,
    negasi_aktif, negasi_intensitas,
    orange_mask,
    temperature, brightness, contrast,
    saturation, highlights, shadows, gamma,
    r_val, g_val, b_val,
    rotation, flip_h, flip_v, zoom, pos_x, pos_y,
    denoise, sharpen, blur, vignette
):
    result = img_bgr.copy()
    result = rotate_image(result, rotation)
    result = flip_image(result, flip_h, flip_v)
    result = zoom_image(result, zoom)
    result = translate_image(result, pos_x, pos_y)
    result = remove_orange_mask(result, strength=orange_mask / 100.0)
    result = apply_negasi(result, aktif=negasi_aktif, intensitas=negasi_intensitas)
    result = adjust_rgb_channels(result, r_val, g_val, b_val)
    result = adjust_temperature(result, temperature)
    result = adjust_brightness(result, brightness)
    result = adjust_contrast(result, contrast)
    result = adjust_saturation(result, saturation)
    result = adjust_highlights_shadows(result, highlights, shadows)
    result = adjust_gamma(result, gamma)
    result = apply_denoise(result, denoise)
    result = apply_sharpen(result, sharpen)
    result = apply_blur(result, blur)
    result = apply_vignette(result, vignette)
    return result


# ─────────────────────────────────────────────
# CELL 6 — VISUALISASI HISTOGRAM (MATPLOTLIB)
# ─────────────────────────────────────────────

def generate_histogram(img_bgr, title="Histogram"):
    fig, ax = plt.subplots(figsize=(6, 2.5))
    fig.patch.set_facecolor('#161410')
    ax.set_facecolor('#0e0d0b')

    colors = {
        'Blue'  : ('#70a8e0', 0),
        'Green' : ('#7ec87e', 1),
        'Red'   : ('#e07070', 2),
    }

    for label, (color, ch) in colors.items():
        hist = cv2.calcHist([img_bgr], [ch], None, [256], [0, 256])
        hist = hist.flatten()
        ax.fill_between(range(256), hist, alpha=0.4, color=color, label=label)
        ax.plot(range(256), hist, alpha=0.8, color=color, linewidth=0.8)

    ax.set_xlim(0, 255)
    ax.set_title(title, color='#8a7d6a', fontsize=9, pad=6)
    ax.tick_params(colors='#4a4035', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#252018')
    ax.legend(fontsize=7, facecolor='#1e1b16', edgecolor='#252018',
              labelcolor='#8a7d6a', loc='upper right')
    plt.tight_layout(pad=0.5)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, facecolor='#161410')
    buf.seek(0)
    plt.close(fig)
    hist_img = Image.open(buf).copy()
    buf.close()
    return hist_img


def get_center_matrix(img_bgr):
    if img_bgr is None:
        return np.zeros((5, 5), dtype=np.uint8)
    h, w = img_bgr.shape[:2]
    cy, cx = h // 2, w // 2
    y_start = max(0, cy - 2)
    y_end   = min(h, cy + 3)
    x_start = max(0, cx - 2)
    x_end   = min(w, cx + 3)
    patch = img_bgr[y_start:y_end, x_start:x_end]
    gray_patch = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    if gray_patch.shape[0] < 5 or gray_patch.shape[1] < 5:
        padded = np.zeros((5, 5), dtype=np.uint8)
        ph, pw = gray_patch.shape
        padded[:ph, :pw] = gray_patch
        return padded
    return gray_patch


def generate_matrix_html(matrix, matrix_compare=None, is_after=False):
    title = "Matriks after(Hasil)" if is_after else "Matriks before (Negatif)"
    html = []
    html.append(f"<div style='text-align: center; font-size: 11px; color: #8a7d6a; margin-bottom: 6px; font-weight: bold; letter-spacing: 1px;'>{title}</div>")
    html.append("<table style='border-collapse: collapse; margin: 0 auto; font-family: \"DM Mono\", monospace; font-size: 12px; color: #f0e8d8; background-color: #0e0d0b; border: 2px solid rgba(245, 166, 35, 0.3); width: 100%; max-width: 220px; text-align: center; box-shadow: 0 4px 15px rgba(0,0,0,0.5); border-radius: 6px; overflow: hidden;'>")
    for r in range(5):
        html.append("  <tr>")
        for c in range(5):
            val = matrix[r, c]
            style_str = "style='border: 1px solid rgba(245, 166, 35, 0.15); width: 40px; height: 40px; padding: 0; vertical-align: middle; text-align: center;'"
            if is_after and matrix_compare is not None:
                if val != matrix_compare[r, c]:
                    style_str = "style='border: 1.5px solid #f76b1c; width: 40px; height: 40px; padding: 0; vertical-align: middle; text-align: center; background: linear-gradient(135deg, #f5a623, #f76b1c) !important; color: #0e0d0b !important; font-weight: bold; box-shadow: inset 0 0 8px rgba(0,0,0,0.4);'"
            html.append(f"    <td {style_str}>{val}</td>")
        html.append("  </tr>")
    html.append("</table>")
    return "\n".join(html)


def generate_placeholder_matrix_html(is_after=False):
    title = "Matriks after (Hasil)" if is_after else "Matriks before(Negatif)"
    html = []
    html.append(f"<div style='text-align: center; font-size: 11px; color: #8a7d6a; margin-bottom: 6px; font-weight: bold; letter-spacing: 1px;'>{title}</div>")
    html.append("<table style='border-collapse: collapse; margin: 0 auto; font-family: \"DM Mono\", monospace; font-size: 12px; color: #4a4035; background-color: #0e0d0b; border: 2px solid rgba(245, 166, 35, 0.15); width: 100%; max-width: 220px; text-align: center; box-shadow: 0 4px 15px rgba(0,0,0,0.3); border-radius: 6px; overflow: hidden;'>")
    for r in range(5):
        html.append("  <tr>")
        for c in range(5):
            html.append("    <td style='border: 1px solid rgba(245, 166, 35, 0.08); width: 40px; height: 40px; padding: 0; vertical-align: middle; text-align: center;'>-</td>")
        html.append("  </tr>")
    html.append("</table>")
    return "\n".join(html)


# ─────────────────────────────────────────────
# CELL 6B — FUNGSI RUMUS OTOMATIS  ← BARU
# ─────────────────────────────────────────────

def generate_formula_html(
    tab_index,
    negasi_aktif=True, negasi_intensitas=100,
    orange_mask=80,
    temperature=0, brightness=0, contrast=0,
    saturation=0, highlights=0, shadows=0, gamma=1.0,
    r_val=0, g_val=0, b_val=0,
    rotation=0, flip_h=False, flip_v=False,
    zoom=100, pos_x=0, pos_y=0,
    denoise=0, sharpen=0, blur=0, vignette=0
):
    """
    Menghasilkan HTML rumus matematis berdasarkan tab aktif dan nilai parameter saat ini.
    Rumus yang aktif (nilai berubah dari default) ditampilkan lebih terang + dot amber.
    tab_index: 0=Upload, 1=Konversi&Mask, 2=Warna&Cahaya, 3=ChannelRGB, 4=Geometri, 5=Filter
    """

    def formula_block(title, formula, note="", category="OPERASI TITIK", active=False):
        border_color = "#f5a623" if active else "rgba(245,166,35,0.15)"
        bg_color     = "rgba(245,166,35,0.07)" if active else "rgba(255,255,255,0.01)"
        opacity      = "1" if active else "0.38"
        cat_colors   = {
            "ARITMATIKA"   : "#7ec87e",
            "OPERASI TITIK": "#f5a623",
            "GEOMETRI"     : "#70a8e0",
            "SPASIAL"      : "#c87ec8",
        }
        cat_color  = cat_colors.get(category, "#8a7d6a")
        active_dot = (
            "<span style='display:inline-block;width:7px;height:7px;"
            "background:#f5a623;border-radius:50%;margin-right:6px;"
            "box-shadow:0 0 6px #f5a62388;flex-shrink:0;'></span>"
        ) if active else "<span style='display:inline-block;width:7px;height:7px;margin-right:6px;flex-shrink:0;'></span>"

        note_html = (
            f"<div style='font-family:DM Mono,monospace;font-size:10px;"
            f"color:#8a7d6a;margin-top:5px;line-height:1.6;'>{note}</div>"
        ) if note else ""

        return (
            f"<div style='margin-bottom:10px;border:1px solid {border_color};"
            f"border-radius:8px;padding:11px 13px;background:{bg_color};"
            f"opacity:{opacity};transition:all 0.25s;'>"
            f"  <div style='display:flex;align-items:center;margin-bottom:7px;'>"
            f"    {active_dot}"
            f"    <span style='font-family:DM Mono,monospace;font-size:11px;"
            f"font-weight:600;color:#f0e8d8;letter-spacing:0.4px;flex:1;'>{title}</span>"
            f"    <span style='font-family:DM Mono,monospace;font-size:9px;color:{cat_color};"
            f"background:{cat_color}22;padding:2px 7px;border-radius:10px;"
            f"border:1px solid {cat_color}44;letter-spacing:1px;white-space:nowrap;margin-left:6px;'>{category}</span>"
            f"  </div>"
            f"  <div style='font-family:DM Mono,monospace;font-size:12px;color:#f5c842;"
            f"background:rgba(0,0,0,0.38);padding:7px 11px;border-radius:6px;"
            f"letter-spacing:0.3px;border-left:3px solid #f5a62355;"
            f"word-break:break-all;line-height:1.55;'>{formula}</div>"
            f"  {note_html}"
            f"</div>"
        )

    def section_header(text):
        return (
            f"<div style='font-family:DM Mono,monospace;font-size:10px;"
            f"letter-spacing:2px;color:#8a7d6a;padding:6px 0 5px;"
            f"border-bottom:1px solid rgba(245,166,35,0.1);"
            f"margin-bottom:10px;text-transform:uppercase;'>{text}</div>"
        )

    parts = [
        "<div style='padding:4px 2px;'>",
        "<div style='font-family:DM Mono,monospace;font-size:10px;color:#8a7d6a;"
        "letter-spacing:2px;text-align:center;margin-bottom:12px;padding-bottom:8px;"
        "border-bottom:1px solid rgba(245,166,35,0.1);text-transform:uppercase;'>"
        "📐 Rumus Yang Digunakan</div>",
    ]

    # ── TAB 0 — UPLOAD ──────────────────────────────────
    if tab_index == 0:
        parts.append(section_header("Format Data Citra"))
        parts.append(formula_block(
            "Format Piksel BGR (OpenCV)",
            "img[y, x] = [B, G, R]  →  dtype: uint8  →  range [0, 255]",
            "OpenCV menggunakan urutan BGR, bukan RGB seperti PIL/Matplotlib.",
            "ARITMATIKA", True
        ))
        parts.append(formula_block(
            "Konversi PIL ↔ OpenCV",
            "BGR = RGB[:, :, ::-1]  |  np.array(pil) → cvtColor(RGB→BGR)",
            "Setiap gambar dikonversi sebelum & sesudah diproses OpenCV.",
            "ARITMATIKA", True
        ))

    # ── TAB 1 — KONVERSI & MASK ─────────────────────────
    elif tab_index == 1:
        parts.append(section_header("Konversi & Orange Mask"))

        negasi_active = negasi_aktif and negasi_intensitas > 0
        alpha_n = negasi_intensitas / 100.0
        parts.append(formula_block(
            "Negasi / Inversi Warna",
            f"P_neg = 255 − P",
            f"P_hasil = {alpha_n:.2f} × P_neg + {1-alpha_n:.2f} × P  "
            f"| intensitas={negasi_intensitas}%  aktif={'✓' if negasi_aktif else '✗'}",
            "OPERASI TITIK", negasi_active
        ))

        orange_active = orange_mask > 0
        s = orange_mask / 100.0
        parts.append(formula_block(
            "Hapus Orange Mask — Stretch Per Channel",
            "P_norm = (P − P₁%) ÷ (P₉₉% − P₁%) × 255",
            f"P_hasil = {s:.2f} × P_norm + {1-s:.2f} × P  "
            f"| strength={orange_mask}%  {'← AKTIF' if orange_active else '← TIDAK AKTIF'}",
            "ARITMATIKA", orange_active
        ))
        parts.append(formula_block(
            "Percentile sebagai Estimasi Mask",
            "mask_low = percentile(channel, 1%)  |  mask_high = percentile(channel, 99%)",
            "Menggunakan percentile 1%–99% agar robust terhadap piksel outlier.",
            "ARITMATIKA", orange_active
        ))

    # ── TAB 2 — WARNA & CAHAYA ──────────────────────────
    elif tab_index == 2:
        parts.append(section_header("Warna & Cahaya"))

        parts.append(formula_block(
            "Brightness (Kecerahan)",
            f"P_baru = P + ({brightness:+d})",
            f"Tambah konstanta ke semua channel  | nilai={brightness:+d}  "
            f"{'← AKTIF' if brightness != 0 else '← NETRAL (tidak berubah)'}",
            "ARITMATIKA", brightness != 0
        ))
        alpha_c = 1.0 + contrast / 100.0
        parts.append(formula_block(
            "Contrast (Kontras)",
            f"P_baru = {alpha_c:.2f} × (P − 128) + 128",
            f"α = 1 + ({contrast}/100) = {alpha_c:.2f}  "
            f"{'← AKTIF' if contrast != 0 else '← NETRAL'}",
            "OPERASI TITIK", contrast != 0
        ))
        shift_t = abs(temperature) * 0.5
        temp_desc = "Hangat: R↑ B↓" if temperature > 0 else "Dingin: R↓ B↑" if temperature < 0 else "Netral"
        parts.append(formula_block(
            "Temperature (Suhu Warna)",
            f"R {'+ ' if temperature >= 0 else '− '}{shift_t:.1f}  |  B {'− ' if temperature >= 0 else '+ '}{shift_t:.1f}",
            f"{temp_desc}  | shift={shift_t:.1f}  "
            f"{'← AKTIF' if temperature != 0 else '← NETRAL'}",
            "ARITMATIKA", temperature != 0
        ))
        alpha_s = 1.0 + saturation / 100.0
        parts.append(formula_block(
            "Saturation (Ruang HSV)",
            f"S_baru = S × {alpha_s:.2f}",
            f"BGR→HSV, kalikan channel S dengan α={alpha_s:.2f}, HSV→BGR  "
            f"{'← AKTIF' if saturation != 0 else '← NETRAL'}",
            "OPERASI TITIK", saturation != 0
        ))
        parts.append(formula_block(
            "Highlights & Shadows",
            "Hi_mask = clip((L−0.5)×2, 0,1)  |  Sh_mask = clip((0.5−L)×2, 0,1)",
            f"P_hasil = P + ({highlights:+d}×Hi) + ({shadows:+d}×Sh)  "
            f"| L = luminance  "
            f"{'← AKTIF' if highlights != 0 or shadows != 0 else '← NETRAL'}",
            "OPERASI TITIK", highlights != 0 or shadows != 0
        ))
        inv_g = 1.0 / gamma if gamma != 0 else 1.0
        parts.append(formula_block(
            "Gamma Correction (LUT 256 nilai)",
            f"P_baru = (P ÷ 255) ^ (1 ÷ {gamma:.2f}) × 255",
            f"exp = 1/γ = {inv_g:.3f}  | "
            f"{'γ < 1 → lebih terang' if gamma < 1 else 'γ > 1 → lebih gelap' if gamma > 1 else 'γ = 1 → tidak berubah'}",
            "OPERASI TITIK", gamma != 1.0
        ))

    # ── TAB 3 — CHANNEL RGB ─────────────────────────────
    elif tab_index == 3:
        parts.append(section_header("Koreksi Channel RGB (Per Channel)"))

        parts.append(formula_block(
            "Red Channel",
            f"R_baru = R + ({r_val:+d})",
            f"img[:,:,2] += {r_val}  (index BGR=2)  "
            f"{'← AKTIF' if r_val != 0 else '← NETRAL'}",
            "ARITMATIKA", r_val != 0
        ))
        parts.append(formula_block(
            "Green Channel",
            f"G_baru = G + ({g_val:+d})",
            f"img[:,:,1] += {g_val}  (index BGR=1)  "
            f"{'← AKTIF' if g_val != 0 else '← NETRAL'}",
            "ARITMATIKA", g_val != 0
        ))
        parts.append(formula_block(
            "Blue Channel",
            f"B_baru = B + ({b_val:+d})",
            f"img[:,:,0] += {b_val}  (index BGR=0)  "
            f"{'← AKTIF' if b_val != 0 else '← NETRAL'}",
            "ARITMATIKA", b_val != 0
        ))
        parts.append(formula_block(
            "Clip — Overflow Protection",
            "P_final = clip(P_baru, 0, 255)",
            "Semua channel di-clip ke [0,255] setelah operasi aritmatika.",
            "ARITMATIKA", r_val != 0 or g_val != 0 or b_val != 0
        ))

    # ── TAB 4 — GEOMETRI ────────────────────────────────
    elif tab_index == 4:
        parts.append(section_header("Transformasi Geometri"))

        import math
        rad  = math.radians(rotation)
        cos_ = round(math.cos(rad), 3)
        sin_ = round(math.sin(rad), 3)
        parts.append(formula_block(
            "Rotasi — Rotation Matrix 2×3",
            f"M = [[{cos_}, −{sin_}, tx], [{sin_}, {cos_}, ty]]",
            f"getRotationMatrix2D(center, {rotation}°, scale=1.0) → warpAffine  "
            f"{'← AKTIF' if rotation != 0 else '← NETRAL'}",
            "GEOMETRI", rotation != 0
        ))

        if flip_h and flip_v:
            flip_desc = "Horizontal + Vertikal  (flipCode = −1)"
            flip_formula = "dst[y,x] = src[H−1−y, W−1−x]"
        elif flip_h:
            flip_desc = "Horizontal  (flipCode = 1)"
            flip_formula = "dst[y,x] = src[y, W−1−x]"
        elif flip_v:
            flip_desc = "Vertikal  (flipCode = 0)"
            flip_formula = "dst[y,x] = src[H−1−y, x]"
        else:
            flip_desc    = "Tidak aktif"
            flip_formula = "dst = src  (tidak ada perubahan)"
        parts.append(formula_block(
            "Flip / Pencerminan",
            flip_formula,
            f"Mode: {flip_desc}  "
            f"{'← AKTIF' if flip_h or flip_v else '← TIDAK AKTIF'}",
            "GEOMETRI", flip_h or flip_v
        ))

        zoom_mode = "Zoom In: crop tengah → resize" if zoom > 100 else "Zoom Out: resize → tempatkan di canvas" if zoom < 100 else "Normal (100%)"
        alpha_z   = zoom / 100.0
        parts.append(formula_block(
            "Zoom (Scale Transform)",
            f"α = {alpha_z:.2f}  |  new_size = original ÷ α  (zoom in)  /  × α  (zoom out)",
            f"Mode: {zoom_mode}  | factor={zoom}%  "
            f"{'← AKTIF' if zoom != 100 else '← NETRAL'}",
            "GEOMETRI", zoom != 100
        ))

        parts.append(formula_block(
            "Translasi — Translation Matrix",
            f"M = [[1, 0, dx], [0, 1, dy]]  |  dx={pos_x}%w  dy={pos_y}%h",
            f"dx = w×{pos_x}/100  |  dy = h×{pos_y}/100  → warpAffine  "
            f"{'← AKTIF' if pos_x != 0 or pos_y != 0 else '← NETRAL'}",
            "GEOMETRI", pos_x != 0 or pos_y != 0
        ))

    # ── TAB 5 — FILTER & EFEK ───────────────────────────
    elif tab_index == 5:
        parts.append(section_header("Filter Spasial"))

        parts.append(formula_block(
            "Denoise — Non-Local Means (NLM)",
            f"P̂(x) = Σ w(x,y) · P(y)  |  h={int(denoise)}",
            f"w(x,y) = exp(−‖P(Nx)−P(Ny)‖² / h²)  | window=7×7  search=21×21  "
            f"{'← AKTIF' if denoise > 0 else '← TIDAK AKTIF'}",
            "SPASIAL", denoise > 0
        ))
        alpha_sh = sharpen / 5.0
        parts.append(formula_block(
            "Sharpen — Unsharp Masking",
            f"P_sharp = {1+alpha_sh:.2f}×P − {alpha_sh:.2f}×GaussianBlur(P, σ=3)",
            f"addWeighted(P, {1+alpha_sh:.2f}, blur, {-alpha_sh:.2f}, 0)  "
            f"{'← AKTIF' if sharpen > 0 else '← TIDAK AKTIF'}",
            "SPASIAL", sharpen > 0
        ))
        k_blur = int(blur) * 2 + 1
        parts.append(formula_block(
            "Gaussian Blur",
            f"G(x,y) = 1÷(2πσ²) · exp(−(x²+y²)÷2σ²)  |  kernel={k_blur}×{k_blur}",
            f"GaussianBlur(img, ({k_blur},{k_blur}), 0)  "
            f"{'← AKTIF' if blur > 0 else '← TIDAK AKTIF'}",
            "SPASIAL", blur > 0
        ))
        sigma_v = max(50, int((1 - vignette / 150.0) * 500))
        parts.append(formula_block(
            "Vignette — Gaussian Mask 2D",
            f"mask = Gy(h,σ={sigma_v}) × Gx(w,σ={sigma_v})ᵀ  →  normalize",
            f"P_hasil = P × (mask ÷ max(mask))  | strength={vignette}%  "
            f"{'← AKTIF' if vignette > 0 else '← TIDAK AKTIF'}",
            "SPASIAL", vignette > 0
        ))

    parts.append("</div>")
    return "\n".join(parts)


def generate_placeholder_formula_html():
    """Placeholder rumus sebelum gambar diupload."""
    return (
        "<div style='padding:16px;text-align:center;'>"
        "<div style='font-family:DM Mono,monospace;font-size:10px;"
        "color:rgba(245,166,35,0.3);letter-spacing:2px;margin-bottom:14px;"
        "text-transform:uppercase;'>📐 Rumus Yang Digunakan</div>"
        "<div style='border:1px dashed rgba(245,166,35,0.15);border-radius:8px;padding:20px;'>"
        "<div style='font-size:26px;margin-bottom:8px;opacity:0.25;'>∫ƒ(x)dx</div>"
        "<div style='font-family:DM Mono,monospace;font-size:11px;"
        "color:rgba(245,166,35,0.3);letter-spacing:1px;'>"
        "Upload gambar untuk melihat rumus aktif</div>"
        "</div></div>"
    )


# ─────────────────────────────────────────────
# CELL 7 — HELPER KONVERSI
# ─────────────────────────────────────────────

def pil_to_bgr(pil_img):
    rgb = np.array(pil_img)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

def bgr_to_pil(bgr_img):
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


# ─────────────────────────────────────────────
# CELL 8 — FUNGSI GRADIO
# ─────────────────────────────────────────────

def proses_otomatis(input_image):
    if input_image is None:
        return None, None, None
    img_bgr   = pil_to_bgr(input_image)
    hasil_bgr = pipeline_otomatis(img_bgr)
    hasil_pil = bgr_to_pil(hasil_bgr)
    hist_before = generate_histogram(img_bgr,   "Histogram — Before (Negatif)")
    hist_after  = generate_histogram(hasil_bgr, "Histogram — After (Hasil)")
    return hasil_pil, hist_before, hist_after


def proses_manual(
    input_image,
    negasi_aktif, negasi_intensitas,
    orange_mask,
    temperature, brightness, contrast,
    saturation, highlights, shadows, gamma,
    r_val, g_val, b_val,
    rotation, flip_h, flip_v, zoom, pos_x, pos_y,
    denoise, sharpen, blur, vignette
):
    """Handler untuk mode manual — sekarang mengembalikan rumus per tab."""

    # ── kwargs untuk generate_formula_html ──
    formula_kwargs = dict(
        negasi_aktif=negasi_aktif, negasi_intensitas=negasi_intensitas,
        orange_mask=orange_mask,
        temperature=temperature, brightness=brightness, contrast=contrast,
        saturation=saturation, highlights=highlights, shadows=shadows, gamma=gamma,
        r_val=r_val, g_val=g_val, b_val=b_val,
        rotation=rotation, flip_h=flip_h, flip_v=flip_v,
        zoom=zoom, pos_x=pos_x, pos_y=pos_y,
        denoise=denoise, sharpen=sharpen, blur=blur, vignette=vignette
    )

    if input_image is None:
        ph_bef     = generate_placeholder_matrix_html(is_after=False)
        ph_aft     = generate_placeholder_matrix_html(is_after=True)
        ph_formula = generate_placeholder_formula_html()
        empty = (None, None, None, ph_bef, ph_aft, ph_formula)
        return empty * 6

    img_pil = (input_image.get("composite") or input_image.get("background")) \
              if isinstance(input_image, dict) else input_image

    if img_pil is None:
        ph_bef     = generate_placeholder_matrix_html(is_after=False)
        ph_aft     = generate_placeholder_matrix_html(is_after=True)
        ph_formula = generate_placeholder_formula_html()
        empty = (None, None, None, ph_bef, ph_aft, ph_formula)
        return empty * 6

    img_bgr = pil_to_bgr(img_pil)

    hasil_bgr = pipeline_manual(
        img_bgr,
        negasi_aktif, negasi_intensitas,
        orange_mask,
        temperature, brightness, contrast,
        saturation, highlights, shadows, gamma,
        r_val, g_val, b_val,
        rotation, flip_h, flip_v, zoom, pos_x, pos_y,
        denoise, sharpen, blur, vignette
    )

    hasil_pil   = bgr_to_pil(hasil_bgr)
    hist_before = generate_histogram(img_bgr,   "Histogram — Before (Negatif)")
    hist_after  = generate_histogram(hasil_bgr, "Histogram — After (Hasil)")

    matrix_bef = get_center_matrix(img_bgr)
    matrix_aft = get_center_matrix(hasil_bgr)
    html_bef   = generate_matrix_html(matrix_bef, is_after=False)
    html_aft   = generate_matrix_html(matrix_aft, matrix_compare=matrix_bef, is_after=True)

    # Rumus per tab (tab 0–5)
    f0 = generate_formula_html(0, **formula_kwargs)
    f1 = generate_formula_html(1, **formula_kwargs)
    f2 = generate_formula_html(2, **formula_kwargs)
    f3 = generate_formula_html(3, **formula_kwargs)
    f4 = generate_formula_html(4, **formula_kwargs)
    f5 = generate_formula_html(5, **formula_kwargs)

    return (
        hasil_pil, hist_before, hist_after, html_bef, html_aft, f0,  # Tab 1 — Upload Foto
        hasil_pil, hist_before, hist_after, html_bef, html_aft, f4,  # Tab 2 — Geometri
        hasil_pil, hist_before, hist_after, html_bef, html_aft, f1,  # Tab 3 — Konversi & Mask
        hasil_pil, hist_before, hist_after, html_bef, html_aft, f2,  # Tab 4 — Warna & Cahaya
        hasil_pil, hist_before, hist_after, html_bef, html_aft, f3,  # Tab 5 — Channel RGB
        hasil_pil, hist_before, hist_after, html_bef, html_aft, f5,  # Tab 6 — Filter & Efek
    )


# ─────────────────────────────────────────────
# CELL 9 — CUSTOM CSS & UI GRADIO
# ─────────────────────────────────────────────

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=Bebas+Neue&family=DM+Sans:wght@300;400;500;600&display=swap');

:root {
    --bg:        #0e0d0b;
    --bg2:       #161410;
    --bg3:       #1e1b16;
    --bg4:       #252018;
    --amber:     #f5a623;
    --amber-dim: rgba(245,166,35,0.12);
    --text:      #f0e8d8;
    --text-dim:  #8a7d6a;
    --border:    rgba(245,166,35,0.15);
    --border2:   rgba(245,166,35,0.08);
    --green:     #7ec87e;
    --red:       #e07070;
    --radius:    10px;
}

body, .gradio-container {
    background: var(--bg) !important;
    font-family: 'DM Mono', monospace !important;
    color: var(--text) !important;
    text-transform: uppercase !important;
}

#main-row { align-items: flex-start !important; }

#right-column {
    position: -webkit-sticky !important;
    position: sticky !important;
    top: 20px !important;
    align-self: start !important;
    z-index: 10 !important;
}

#manual-input,
#manual-input .image-container,
#manual-input .image-container img,
#manual-input .image-container canvas,
#manual-input .upload-container,
#manual-input .image-frame,
#manual-input .preview,
#manual-input .box,
#manual-input .bg-white,
#manual-input .bg-gray-50,
#manual-input [data-testid="image"],
#manual-input [data-testid="file-preview"],
#manual-input .sketch-container,
#manual-input div.wrap,
#manual-input button.upload-button,
#manual-input button {
    background-color: var(--bg3) !important;
    color: var(--text) !important;
    text-transform: none !important;
}
#manual-input .upload-container span,
#manual-input .upload-container p,
#manual-input .upload-container div {
    color: var(--text-dim) !important;
}

.film-header {
    text-align: center;
    padding: 28px 0 8px;
    border-bottom: 1px solid var(--border2);
    margin-bottom: 20px;
}

.film-title {
    font-family: 'Bebas Neue', sans-serif !important;
    font-size: 3rem !important;
    letter-spacing: 6px !important;
    background: linear-gradient(135deg, #f5a623, #f76b1c) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    margin: 0 !important;
}

.film-subtitle {
    font-family: 'DM Mono', monospace !important;
    font-size: 11px !important;
    color: var(--text-dim) !important;
    letter-spacing: 3px !important;
    text-transform: uppercase !important;
    margin-top: 4px !important;
}

#custom-tabs, div#custom-tabs {
    border: none !important;
    background: transparent !important;
}

#custom-tabs > div:first-child,
#custom-tabs .tab-nav,
#custom-tabs [role="tablist"] {
    display: flex !important;
    justify-content: space-between !important;
    align-items: center !important;
    flex-wrap: nowrap !important;
    gap: 0 !important;
    background: transparent !important;
    border: none !important;
    border-bottom: 1px solid var(--border2) !important;
    border-radius: 0 !important;
    padding: 0 !important;
    margin: 10px 0 20px 0 !important;
    max-width: none !important;
    width: 100% !important;
    box-shadow: none !important;
}

#custom-tabs > div:first-child button,
#custom-tabs .tab-nav button,
#custom-tabs [role="tab"] {
    font-family: 'DM Mono', monospace !important;
    font-size: 13px !important;
    font-weight: 600 !important;
    color: var(--text-dim) !important;
    background: transparent !important;
    border: none !important;
    border-radius: 0 !important;
    padding: 12px 8px !important;
    letter-spacing: 0.5px !important;
    transition: all 0.25s cubic-bezier(0.4, 0, 0.2, 1) !important;
    border-bottom: 2px solid transparent !important;
    flex: 1 1 0% !important;
    text-align: center !important;
    white-space: nowrap !important;
    text-transform: uppercase !important;
}

#custom-tabs > div:first-child button::after,
#custom-tabs .tab-nav button::after,
#custom-tabs [role="tab"]::after {
    display: none !important;
    content: none !important;
}

#custom-tabs > div:first-child button:hover,
#custom-tabs .tab-nav button:hover,
#custom-tabs [role="tab"]:hover {
    color: var(--text) !important;
    background: rgba(245, 166, 35, 0.05) !important;
}

#custom-tabs > div:first-child button.selected,
#custom-tabs .tab-nav button.selected,
#custom-tabs [role="tab"][aria-selected="true"] {
    color: var(--amber) !important;
    background: transparent !important;
    border-bottom: 2px solid var(--amber) !important;
}

.tabitem {
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    border-radius: 16px !important;
    padding: 24px !important;
    box-shadow: 0 12px 40px rgba(0, 0, 0, 0.5) !important;
    animation: tabFadeIn 0.4s ease-out !important;
}

@keyframes tabFadeIn {
    from { opacity: 0; transform: translateY(8px); }
    to   { opacity: 1; transform: translateY(0); }
}

.gradio-group, .gr-group {
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    border-radius: var(--radius) !important;
    margin-bottom: 8px !important;
}

label, .gr-form label, span.svelte-1gfkn6j {
    font-family: 'DM Mono', monospace !important;
    font-size: 12px !important;
    color: var(--text-dim) !important;
    font-weight: 500 !important;
    text-transform: uppercase !important;
}

input[type=range]    { accent-color: var(--amber) !important; }
input[type=checkbox] { accent-color: var(--amber) !important; }

.btn-amber {
    background: var(--amber) !important;
    color: #0e0d0b !important;
    border: none !important;
    border-radius: 8px !important;
    font-family: 'DM Mono', monospace !important;
    font-weight: 600 !important;
    font-size: 14px !important;
    padding: 12px !important;
    cursor: pointer !important;
    transition: all 0.15s !important;
    width: 100% !important;
    text-transform: uppercase !important;
}

.btn-amber:hover {
    background: #e8941a !important;
    transform: translateY(-1px) !important;
}

.btn-secondary {
    background: transparent !important;
    color: var(--text-dim) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    font-family: 'DM Mono', monospace !important;
    font-weight: 600 !important;
    font-size: 14px !important;
    padding: 12px !important;
    cursor: pointer !important;
    transition: all 0.15s !important;
    width: 100% !important;
    text-transform: uppercase !important;
}

.btn-secondary:hover {
    border-color: var(--amber) !important;
    color: var(--amber) !important;
    background: var(--amber-dim) !important;
    transform: translateY(-1px) !important;
}

.image-container, .gr-image {
    background: var(--bg3) !important;
    border: 1px solid var(--border2) !important;
    border-radius: var(--radius) !important;
}

input[type=number] {
    background: var(--bg3) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 6px !important;
    font-family: 'DM Mono', monospace !important;
    font-size: 12px !important;
}

.section-title {
    font-family: 'DM Mono', monospace !important;
    font-size: 10px !important;
    letter-spacing: 2px !important;
    text-transform: uppercase !important;
    color: var(--text-dim) !important;
    padding: 8px 0 4px !important;
    border-bottom: 1px solid var(--border2) !important;
    margin-bottom: 8px !important;
}

.gr-accordion {
    background: var(--bg2) !important;
    border: 1px solid var(--border2) !important;
    border-radius: var(--radius) !important;
    margin-bottom: 6px !important;
}

.gr-accordion .label {
    color: var(--text-dim) !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    letter-spacing: 1px !important;
    text-transform: uppercase !important;
}

.info-badge {
    display: inline-block;
    font-family: 'DM Mono', monospace;
    font-size: 10px;
    color: var(--amber);
    background: var(--amber-dim);
    border: 1px solid var(--border);
    padding: 3px 10px;
    border-radius: 20px;
    letter-spacing: 1px;
    margin: 2px;
}
"""


# ─────────────────────────────────────────────
# CELL 10 — BUILD UI GRADIO
# ─────────────────────────────────────────────

with gr.Blocks(css=CSS, theme=gr.themes.Base(), title="🎞️ Film Roll Converter") as demo:

    gr.HTML("""
    <div class="film-header">
        <h1 class="film-title">🎞 Film Roll Converter</h1>
        <p class="film-subtitle">Sistem Restorasi dan Koreksi Warna Foto Usang · OpenCV · Python · Gradio</p>
    </div>
    """)

    with gr.Tabs(elem_id="custom-tabs"):

        # ═══════════════════════════════════════
        # TAB 1 — UPLOAD FOTO
        # ═══════════════════════════════════════
        with gr.Tab("📤  Upload Foto"):
            with gr.Row():
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<div class="section-title">🖼️ Upload Foto</div>')
                    manual_input = gr.ImageEditor(
                        type="pil",
                        label="Scan Negatif Film",
                        height=300,
                        elem_id="manual-input"
                    )
                    # ── RUMUS TAB 1 ──
                    gr.HTML('<div class="section-title" style="margin-top:10px">📐 Rumus Matematis</div>')
                    manual_formula_1 = gr.HTML(value=generate_placeholder_formula_html())
                with gr.Column(scale=1):
                    gr.HTML('<div class="section-title">🖼️ Before / After</div>')
                    with gr.Row():
                        manual_before_1 = gr.Image(type="pil", label="Before — Negatif", height=250, interactive=False)
                        manual_output_1 = gr.Image(type="pil", label="After — Hasil",    height=250, interactive=False)
                    gr.HTML('<div class="section-title" style="margin-top:10px">📊 Analisis Histogram RGB</div>')
                    with gr.Row():
                        manual_hist_before_1 = gr.Image(type="pil", label="Histogram — Before", height=160)
                        manual_hist_after_1  = gr.Image(type="pil", label="Histogram — After",  height=160)
                    gr.HTML('<div class="section-title" style="margin-top:10px">🔢 Matriks Intensitas Pixel (5×5 Center)</div>')
                    with gr.Row():
                        manual_matrix_before_1 = gr.HTML(value=generate_placeholder_matrix_html(is_after=False))
                        manual_matrix_after_1  = gr.HTML(value=generate_placeholder_matrix_html(is_after=True))

        # ═══════════════════════════════════════
        # TAB 5 — GEOMETRI
        # ═══════════════════════════════════════
        with gr.Tab("📐  Geometri"):
            with gr.Row():
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<div class="section-title">🖼️ Geometri</div>')
                    with gr.Accordion("📐  Geometri", open=True) as acc_geometri:
                        rotation = gr.Slider(-180, 180, value=0, step=0.5, label="🔃 Rotasi (°)")
                        with gr.Row():
                            flip_h = gr.Checkbox(value=False, label="↔️ Flip Horizontal")
                            flip_v = gr.Checkbox(value=False, label="↕️ Flip Vertikal")
                        zoom  = gr.Slider(50, 200, value=100, step=5,  label="🔍 Zoom In/Out")
                        with gr.Row():
                            pos_x = gr.Slider(-50, 50, value=0, step=1, label="↔️ Posisi X")
                            pos_y = gr.Slider(-50, 50, value=0, step=1, label="↕️ Posisi Y")
                    # ── RUMUS TAB 5 ──
                    gr.HTML('<div class="section-title" style="margin-top:10px">📐 Rumus Matematis</div>')
                    manual_formula_5 = gr.HTML(value=generate_placeholder_formula_html())
                with gr.Column(scale=1):
                    gr.HTML('<div class="section-title">🖼️ Before / After</div>')
                    with gr.Row():
                        manual_before_5 = gr.Image(type="pil", label="Before — Negatif", height=250, interactive=False)
                        manual_output_5 = gr.Image(type="pil", label="After — Hasil",    height=250, interactive=False)
                    gr.HTML('<div class="section-title" style="margin-top:10px">📊 Analisis Histogram RGB</div>')
                    with gr.Row():
                        manual_hist_before_5 = gr.Image(type="pil", label="Histogram — Before", height=160)
                        manual_hist_after_5  = gr.Image(type="pil", label="Histogram — After",  height=160)
                    gr.HTML('<div class="section-title" style="margin-top:10px">🔢 Matriks Intensitas Pixel (5×5 Center)</div>')
                    with gr.Row():
                        manual_matrix_before_5 = gr.HTML(value=generate_placeholder_matrix_html(is_after=False))
                        manual_matrix_after_5  = gr.HTML(value=generate_placeholder_matrix_html(is_after=True))

        # ═══════════════════════════════════════
        # TAB 2 — KONVERSI & MASK
        # ═══════════════════════════════════════
        with gr.Tab("🔄  Konversi & Mask"):
            with gr.Row():
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<div class="section-title">🖼️ Konversi & Mask</div>')
                    with gr.Accordion("🔄  Negasi", open=True) as acc_negasi:
                        negasi_aktif      = gr.Checkbox(value=False,  label="Aktifkan Negasi")
                        negasi_intensitas = gr.Slider(0, 100, value=0, step=1, label="Intensitas Negasi")
                    with gr.Accordion("🟠  Orange Mask", open=True) as acc_orange:
                        orange_mask = gr.Slider(0, 100, value=0, step=1, label="Kekuatan Koreksi")
                    # ── RUMUS TAB 2 ──
                    gr.HTML('<div class="section-title" style="margin-top:10px">📐 Rumus Matematis</div>')
                    manual_formula_2 = gr.HTML(value=generate_placeholder_formula_html())
                with gr.Column(scale=1):
                    gr.HTML('<div class="section-title">🖼️ Before / After</div>')
                    with gr.Row():
                        manual_before_2 = gr.Image(type="pil", label="Before — Negatif", height=250, interactive=False)
                        manual_output_2 = gr.Image(type="pil", label="After — Hasil",    height=250, interactive=False)
                    gr.HTML('<div class="section-title" style="margin-top:10px">📊 Analisis Histogram RGB</div>')
                    with gr.Row():
                        manual_hist_before_2 = gr.Image(type="pil", label="Histogram — Before", height=160)
                        manual_hist_after_2  = gr.Image(type="pil", label="Histogram — After",  height=160)
                    gr.HTML('<div class="section-title" style="margin-top:10px">🔢 Matriks Intensitas Pixel (5×5 Center)</div>')
                    with gr.Row():
                        manual_matrix_before_2 = gr.HTML(value=generate_placeholder_matrix_html(is_after=False))
                        manual_matrix_after_2  = gr.HTML(value=generate_placeholder_matrix_html(is_after=True))

        # ═══════════════════════════════════════
        # TAB 3 — WARNA & CAHAYA
        # ═══════════════════════════════════════
        with gr.Tab("💡  Warna & Cahaya"):
            with gr.Row():
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<div class="section-title">🖼️ Warna & Cahaya</div>')
                    with gr.Accordion("💡  Cahaya & Warna", open=True) as acc_cahaya:
                        temperature = gr.Slider(-100, 100, value=0,    step=1,    label="🌡️ Temperature")
                        brightness  = gr.Slider(-100, 100, value=0,    step=1,    label="☀️ Brightness")
                        contrast    = gr.Slider(-100, 100, value=0,    step=1,    label="🎚️ Contrast")
                        saturation  = gr.Slider(-100, 100, value=0,    step=1,    label="🎨 Saturation")
                        highlights  = gr.Slider(-100, 100, value=0,    step=1,    label="🔆 Highlights")
                        shadows     = gr.Slider(-100, 100, value=0,    step=1,    label="🌑 Shadows")
                        gamma       = gr.Slider(0.1,  3.0, value=1.0,  step=0.05, label="🎭 Gamma")
                    # ── RUMUS TAB 3 ──
                    gr.HTML('<div class="section-title" style="margin-top:10px">📐 Rumus Matematis</div>')
                    manual_formula_3 = gr.HTML(value=generate_placeholder_formula_html())
                with gr.Column(scale=1):
                    gr.HTML('<div class="section-title">🖼️ Before / After</div>')
                    with gr.Row():
                        manual_before_3 = gr.Image(type="pil", label="Before — Negatif", height=250, interactive=False)
                        manual_output_3 = gr.Image(type="pil", label="After — Hasil",    height=250, interactive=False)
                    gr.HTML('<div class="section-title" style="margin-top:10px">📊 Analisis Histogram RGB</div>')
                    with gr.Row():
                        manual_hist_before_3 = gr.Image(type="pil", label="Histogram — Before", height=160)
                        manual_hist_after_3  = gr.Image(type="pil", label="Histogram — After",  height=160)
                    gr.HTML('<div class="section-title" style="margin-top:10px">🔢 Matriks Intensitas Pixel (5×5 Center)</div>')
                    with gr.Row():
                        manual_matrix_before_3 = gr.HTML(value=generate_placeholder_matrix_html(is_after=False))
                        manual_matrix_after_3  = gr.HTML(value=generate_placeholder_matrix_html(is_after=True))

        # ═══════════════════════════════════════
        # TAB 4 — CHANNEL RGB
        # ═══════════════════════════════════════
        with gr.Tab("🎨  Channel RGB"):
            with gr.Row():
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<div class="section-title">🖼️ Channel RGB</div>')
                    with gr.Accordion("🎨  Channel RGB", open=True) as acc_channel:
                        r_val = gr.Slider(-100, 100, value=0, step=1, label="🔴 Red Channel")
                        g_val = gr.Slider(-100, 100, value=0, step=1, label="🟢 Green Channel")
                        b_val = gr.Slider(-100, 100, value=0, step=1, label="🔵 Blue Channel")
                    # ── RUMUS TAB 4 ──
                    gr.HTML('<div class="section-title" style="margin-top:10px">📐 Rumus Matematis</div>')
                    manual_formula_4 = gr.HTML(value=generate_placeholder_formula_html())
                with gr.Column(scale=1):
                    gr.HTML('<div class="section-title">🖼️ Before / After</div>')
                    with gr.Row():
                        manual_before_4 = gr.Image(type="pil", label="Before — Negatif", height=250, interactive=False)
                        manual_output_4 = gr.Image(type="pil", label="After — Hasil",    height=250, interactive=False)
                    gr.HTML('<div class="section-title" style="margin-top:10px">📊 Analisis Histogram RGB</div>')
                    with gr.Row():
                        manual_hist_before_4 = gr.Image(type="pil", label="Histogram — Before", height=160)
                        manual_hist_after_4  = gr.Image(type="pil", label="Histogram — After",  height=160)
                    gr.HTML('<div class="section-title" style="margin-top:10px">🔢 Matriks Intensitas Pixel (5×5 Center)</div>')
                    with gr.Row():
                        manual_matrix_before_4 = gr.HTML(value=generate_placeholder_matrix_html(is_after=False))
                        manual_matrix_after_4  = gr.HTML(value=generate_placeholder_matrix_html(is_after=True))

        # ═══════════════════════════════════════
        # TAB 6 — FILTER & EFEK
        # ═══════════════════════════════════════
        with gr.Tab("🌀  Filter & Efek"):
            with gr.Row():
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<div class="section-title">🖼️ Filter & Efek</div>')
                    with gr.Accordion("🌀  Filter Spasial", open=True) as acc_spasial:
                        denoise  = gr.Slider(0, 20,  value=0, step=1,   label="🌫️ Denoise")
                        sharpen  = gr.Slider(0, 10,  value=0, step=0.5, label="🔪 Sharpen")
                        blur     = gr.Slider(0, 10,  value=0, step=1,   label="💧 Blur")
                        vignette = gr.Slider(0, 100, value=0, step=1,   label="🌑 Vignette")
                    # ── RUMUS TAB 6 ──
                    gr.HTML('<div class="section-title" style="margin-top:10px">📐 Rumus Matematis</div>')
                    manual_formula_6 = gr.HTML(value=generate_placeholder_formula_html())
                with gr.Column(scale=1):
                    gr.HTML('<div class="section-title">🖼️ Before / After</div>')
                    with gr.Row():
                        manual_before_6 = gr.Image(type="pil", label="Before — Negatif", height=250, interactive=False)
                        manual_output_6 = gr.Image(type="pil", label="After — Hasil",    height=250, interactive=False)
                    gr.HTML('<div class="section-title" style="margin-top:10px">📊 Analisis Histogram RGB</div>')
                    with gr.Row():
                        manual_hist_before_6 = gr.Image(type="pil", label="Histogram — Before", height=160)
                        manual_hist_after_6  = gr.Image(type="pil", label="Histogram — After",  height=160)
                    gr.HTML('<div class="section-title" style="margin-top:10px">🔢 Matriks Intensitas Pixel (5×5)</div>')
                    with gr.Row():
                        manual_matrix_before_6 = gr.HTML(value=generate_placeholder_matrix_html(is_after=False))
                        manual_matrix_after_6  = gr.HTML(value=generate_placeholder_matrix_html(is_after=True))

    # ── ACTION BUTTONS ──
    gr.HTML('<div style="height:12px"></div>')
    with gr.Row():
        btn_manual = gr.Button("Apply Semua Parameter", elem_classes="btn-amber")
        btn_reset  = gr.Button("Reset ke Default",      elem_classes="btn-secondary")

    # ── EVENTS ──

    def update_before(img):
        val = (img.get("composite") or img.get("background")) if isinstance(img, dict) else img
        return val, val, val, val, val, val

    all_manual_inputs = [
        manual_input,
        negasi_aktif, negasi_intensitas,
        orange_mask,
        temperature, brightness, contrast,
        saturation, highlights, shadows, gamma,
        r_val, g_val, b_val,
        rotation, flip_h, flip_v, zoom, pos_x, pos_y,
        denoise, sharpen, blur, vignette
    ]

    # Output: 6 tab × 6 komponen (image, hist_before, hist_after, matrix_bef, matrix_aft, formula)
    all_manual_outputs = [
        manual_output_1, manual_hist_before_1, manual_hist_after_1, manual_matrix_before_1, manual_matrix_after_1, manual_formula_1,
        manual_output_2, manual_hist_before_2, manual_hist_after_2, manual_matrix_before_2, manual_matrix_after_2, manual_formula_2,
        manual_output_3, manual_hist_before_3, manual_hist_after_3, manual_matrix_before_3, manual_matrix_after_3, manual_formula_3,
        manual_output_4, manual_hist_before_4, manual_hist_after_4, manual_matrix_before_4, manual_matrix_after_4, manual_formula_4,
        manual_output_5, manual_hist_before_5, manual_hist_after_5, manual_matrix_before_5, manual_matrix_after_5, manual_formula_5,
        manual_output_6, manual_hist_before_6, manual_hist_after_6, manual_matrix_before_6, manual_matrix_after_6, manual_formula_6,
    ]

    manual_input.change(
        fn=update_before,
        inputs=[manual_input],
        outputs=[manual_before_1, manual_before_2, manual_before_3, manual_before_4, manual_before_5, manual_before_6]
    ).then(
        fn=proses_manual,
        inputs=all_manual_inputs,
        outputs=all_manual_outputs,
        show_progress="hidden"
    )

    btn_manual.click(
        fn=proses_manual,
        inputs=all_manual_inputs,
        outputs=all_manual_outputs
    )

    def reset_all():
        return (
            False, 0,
            0,
            0, 0, 0,
            0, 0, 0,
            1.0,
            0, 0, 0,
            0,
            False, False,
            100,
            0, 0,
            0, 0, 0, 0
        )

    btn_reset.click(
        fn=reset_all,
        inputs=[],
        outputs=[
            negasi_aktif, negasi_intensitas,
            orange_mask,
            temperature, brightness, contrast,
            saturation, highlights, shadows, gamma,
            r_val, g_val, b_val,
            rotation, flip_h, flip_v, zoom, pos_x, pos_y,
            denoise, sharpen, blur, vignette
        ]
    ).then(
        fn=proses_manual,
        inputs=all_manual_inputs,
        outputs=all_manual_outputs,
        show_progress="hidden"
    )

    for component in all_manual_inputs:
        if isinstance(component, gr.Slider):
            component.release(
                fn=proses_manual,
                inputs=all_manual_inputs,
                outputs=all_manual_outputs,
                show_progress="hidden"
            )
        elif component != manual_input:
            component.change(
                fn=proses_manual,
                inputs=all_manual_inputs,
                outputs=all_manual_outputs,
                show_progress="hidden"
            )

    # ── MUTUAL EXCLUSIVITY OF ACCORDIONS ──
    acc_negasi.expand(
        fn=lambda: (gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False)),
        inputs=[],
        outputs=[acc_orange, acc_cahaya, acc_channel, acc_geometri, acc_spasial]
    )
    acc_orange.expand(
        fn=lambda: (gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False)),
        inputs=[],
        outputs=[acc_negasi, acc_cahaya, acc_channel, acc_geometri, acc_spasial]
    )
    acc_cahaya.expand(
        fn=lambda: (gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False)),
        inputs=[],
        outputs=[acc_negasi, acc_orange, acc_channel, acc_geometri, acc_spasial]
    )
    acc_channel.expand(
        fn=lambda: (gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False)),
        inputs=[],
        outputs=[acc_negasi, acc_orange, acc_cahaya, acc_geometri, acc_spasial]
    )
    acc_geometri.expand(
        fn=lambda: (gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False)),
        inputs=[],
        outputs=[acc_negasi, acc_orange, acc_cahaya, acc_channel, acc_spasial]
    )
    acc_spasial.expand(
        fn=lambda: (gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False), gr.Accordion(open=False)),
        inputs=[],
        outputs=[acc_negasi, acc_orange, acc_cahaya, acc_channel, acc_geometri]
    )


# ─────────────────────────────────────────────
# CELL 11 — LAUNCH
# ─────────────────────────────────────────────

demo.launch(share=True, debug=False)

/tmp/ipykernel_7356/83474902.py:1114: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Base(), title="🎞️ Film Roll Converter") as demo:
/tmp/ipykernel_7356/83474902.py:1114: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Base(), title="🎞️ Film Roll Converter") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://69162292f85cfc32f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
